# 🚀 SOTA AI: Kaggle Native Data Pipeline (Feature Extraction + Target Labels)
This notebook is specifically designed to run on **Kaggle**.
1. Click **Add Data** on the top right panel, and search for the datasets: `NASA CMaps`, `PaySim1`, `DataCo SMART Supply Chain`, `M5 Forecasting Accuracy`.
2. Since Kaggle mounts them instantly to `/kaggle/input/`, we skip downloading gigabytes.
3. This script extracts advanced mathematical features (Betti numbers, Log-Signatures) AND actively maps the target prediction labels (RUL, `isFraud`) directly into the final `.csv` outputs so they are ready for model training.

## 1. Install Advanced Mathematical C++ Extensions

In [ ]:
!pip install -q gudhi networkx signatory
print("\n✅ Advanced Mathematical Libraries Installed (GUDHI, Signatory, NetworkX)!")

## 2. Dynamic Kaggle File Discoverer

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import gudhi
import networkx as nx
import signatory
import gc
import IPython.display as display

KAGGLE_INPUT = '/kaggle/input/'
KAGGLE_WORKING = '/kaggle/working/'

print("🔎 Scanning Kaggle Input Directory for your mounted datasets...")
mounted_dirs = os.listdir(KAGGLE_INPUT)
for d in mounted_dirs:
    print(f" - Found Dataset Mount: {d}")


## 3. NASA Telemetry Extract (Log-Signatures + Remaining Useful Life [RUL] Labels)

In [ ]:
def extract_log_signatures(sensor_tensor, depth=3):
    with torch.no_grad():
        return signatory.logsignature(sensor_tensor, depth)

print("---- Processing NASA IoT Telemetry ----")
nasa_train_files = glob.glob(os.path.join(KAGGLE_INPUT, '**', 'train_FD*.txt'), recursive=True)

if not nasa_train_files:
    print("\u26a0\ufe0f Skipping: Could not locate NASA files. Did you 'Add Data' -> 'NASA CMaps'?")
else:
    for file_path in nasa_train_files:
        filename = os.path.basename(file_path)
        dataset_index = filename.replace('train_', '').replace('.txt', '') # e.g., 'FD001'
        
        rul_file_path = os.path.join(os.path.dirname(file_path), f'RUL_{dataset_index}.txt')
        rul_labels = None
        if os.path.exists(rul_file_path):
            rul_labels = pd.read_csv(rul_file_path, header=None)
            print(f"\u2705 Found {len(rul_labels)} ground-truth RUL labels for {dataset_index}!")
            
        out_name = f"features_labeled_logsig_{dataset_index}.csv"
        out_path = os.path.join(KAGGLE_WORKING, out_name)
        if os.path.exists(out_path):
            os.remove(out_path)
            
        print(f"\n-> Engineering {filename} into {out_path}...")
        try:
            df = pd.read_csv(file_path, sep=r'\s+', header=None)
            df = df.dropna(axis=1, how='all')
            engine_ids = df.iloc[:, 0].unique()
            
            for i, eng_id in enumerate(engine_ids):
                engine_data = df[df.iloc[:, 0] == eng_id].iloc[:, 2:].values 
                tensor_data = torch.tensor(engine_data, dtype=torch.float32).unsqueeze(0) 
                
                sig_array = []
                if tensor_data.shape[1] > 1:
                    sig = extract_log_signatures(tensor_data, depth=2)
                    sig_array = sig.squeeze(0).numpy()
                
                row_dict = {f"sig_{j}": val for j, val in enumerate(sig_array)}
                row_dict['engine_id'] = eng_id
                row_dict['cycle_length'] = len(engine_data)
                
                if rul_labels is not None and i < len(rul_labels):
                    row_dict['target_RUL'] = rul_labels.iloc[i, 0]
                else:
                    row_dict['target_RUL'] = np.nan
                    
                pd.DataFrame([row_dict]).to_csv(out_path, mode='a', header=not os.path.exists(out_path), index=False)
                del tensor_data, engine_data
                
            del df
            gc.collect()
            print(f"  \u2705 Extraction Complete! Saved sequentially to {out_path}")
        except Exception as e:
            print(f"  \u274c Error processing: {e}")

## 4. PaySim/Fraud Topology (Betti Numbers + Density Labels)

In [ ]:
def extract_betti_numbers(distance_matrix):
    rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix, max_edge_length=2.0)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=3)
    simplex_tree.persistence()
    return simplex_tree.betti_numbers()

print("\n---- Processing PaySim Fraud Topology ----")
paysim_files = glob.glob(os.path.join(KAGGLE_INPUT, '**', '*paysim*.csv'), recursive=True) + glob.glob(os.path.join(KAGGLE_INPUT, '**', 'PS*.csv'), recursive=True)

if not paysim_files:
    print("\u26a0\ufe0f Skipping: Could not locate PaySim/Fraud file. Please 'Add Data' -> 'PaySim1'.")
else:
    file_path = paysim_files[0]
    out_path = os.path.join(KAGGLE_WORKING, "features_labeled_topological_fraud.csv")
    if os.path.exists(out_path):
        os.remove(out_path)
        
    print(f"-> Steaming {os.path.basename(file_path)} in 50k row topological sub-graphs...")
    try:
        chunk_index = 0
        for chunk_df in pd.read_csv(file_path, chunksize=50000):
            chunk_index += 1
            
            target_col = [col for col in chunk_df.columns if 'fraud' in col.lower()]
            fraud_count = 0
            if target_col:
                fraud_count = float(chunk_df[target_col[0]].sum())
                
            print(f"  Computing Chunk {chunk_index} Topology... (Contains {int(fraud_count)} Fraudulent txns)")
            
            G = nx.from_pandas_edgelist(chunk_df, 'nameOrig', 'nameDest', ['amount'], create_using=nx.Graph())
            components = sorted(nx.connected_components(G), key=len, reverse=True)
            
            betti = [0, 0, 0]
            if components:
                sub_G = G.subgraph(components[0])
                nodes = list(sub_G.nodes())[:150]
                sub_G = sub_G.subgraph(nodes)
                
                dist_matrix = np.zeros((len(nodes), len(nodes)))
                for i, n1 in enumerate(nodes):
                    for j, n2 in enumerate(nodes):
                        if i != j:
                            if sub_G.has_edge(n1, n2):
                                dist_matrix[i, j] = 1.0 / (sub_G[n1][n2]['amount'] + 1e-5)
                            else:
                                dist_matrix[i, j] = 10.0
                
                dist_matrix = (dist_matrix + dist_matrix.T) / 2
                np.fill_diagonal(dist_matrix, 0)
                betti = extract_betti_numbers(dist_matrix)
                
            pd.DataFrame([{
                "graph_chunk_id": chunk_index,
                "betti_0_connected_comps": betti[0] if len(betti) > 0 else 0, 
                "betti_1_rings": betti[1] if len(betti) > 1 else 0,
                "betti_2_voids": betti[2] if len(betti) > 2 else 0,
                "target_fraud_count": fraud_count,
                "target_contains_fraud": int(fraud_count > 0)
            }]).to_csv(out_path, mode='a', header=not os.path.exists(out_path), index=False)
            
            del G, chunk_df
            gc.collect()
    except Exception as e:
        print(f"  \u274c Error processing: {e}")

## 5. Generate Download Links for Extracted CSVs

In [ ]:
import base64
def create_download_link(file_path):
    if not os.path.exists(file_path):
        return f"Did not generate: {os.path.basename(file_path)}"
    
    with open(file_path, 'rb') as f:
        data = f.read()
        
    b64 = base64.b64encode(data).decode('utf-8')
    html = f'<a download="{os.path.basename(file_path)}" href="data:text/csv;base64,{b64}" target="_blank"><b>\U0001f4e5 Download {os.path.basename(file_path)}</b></a>'
    return display.HTML(html)

print("\n\U0001f389 **PIPELINE COMPLETE: Click below to download your labeled mathematical datasets:** \U0001f389\n")

csvs = glob.glob(os.path.join(KAGGLE_WORKING, '*.csv'))
for file in csvs:
    display.display(create_download_link(file))